# Object-Oriented Programming in C++ — CO1

**2310261L.CO.1** — *Develop solutions for real world problems using Object Oriented
Programming.* **[L3]**

This notebook covers **CO1 only**, in depth. The other outcomes are in
[`subjects/04-object-oriented-cpp.md`](subjects/04-object-oriented-cpp.md).

---

## The problem we were solving

The real-world problem: **a processor's workings are invisible, so students memorise the
fetch-decode-execute diagram without ever seeing it happen.**

The solution is a simulator you can step through. That is a substantial program — 4,102
lines across 28 files — and object orientation is what keeps it from collapsing under its
own weight. Everything below is a decision we had to make, and what OOP gave us for it.

---

## Decision 1 · Who owns what

The first question in any object-oriented design is not "what classes shall we have" but
**"who is responsible for what, and who is allowed to touch it."**

![what each class owns](diagrams/oop-classes.png)

| Class | Owns | Responsible for |
|---|---|---|
| `RegisterFile` | the registers and the flags | reads, writes, and reporting what changed this cycle |
| `ALU` | **nothing at all** | given an operation and two values, return a result and flags |
| `Memory` | the addressable store | reads and writes by address, and nothing else |
| `CPU` | registers, ALU, memory, the program | running one step and keeping the machine consistent |
| `Canvas` | the pixel buffer | turning coordinates into pixels |

Two of these are worth dwelling on.

### The ALU owns nothing — deliberately

**What this does.** The ALU's entire public interface. Values go in, a result and flags
come out, and nothing is remembered between calls.

```cpp
AluResult execute(AluOp op, Word a, Word b);
```
<sub>src/core/ALU.h:82</sub>

Because it stores no state, the same inputs always give the same outputs. That means we
can test overflow behaviour on its own, with no processor around it, and know the answer
is not contaminated by something left over from a previous instruction. **Statelessness is
a design choice, not an accident.**

### Memory can only be reached one way

**What this does.** Reads a memory cell. Cells never written come back as zero rather than
as an error, which matches how real memory behaves.

```cpp
Word Memory::read(unsigned int address) {
    address &= 0xFFFFu;
    u16 v = 0;
    cells_.get(address, v);          // absent cells read as zero
    ++readCount_;
    lastAddress_      = address;
    lastWasWrite_     = false;
    touchedThisCycle_ = true;
    return Word(v);
}
```
<sub>src/core/Memory.cpp:5</sub>

The storage itself is private. Nothing outside the class can reach it, so **every access
in the whole program passes through this one function** — which is exactly why it can also
record what was touched, for the display to highlight. Encapsulation is not a formality
here; it is what makes that possible at all.

---

## Decision 2 · Fifteen instructions, one interface

The processor understands fifteen instructions. Each does something different, and more
will be added later.

The obvious approach is a chain of tests inside the processor. We did not take it, and the
reason is worth explaining carefully.

**What this does.** Declares what every instruction must be able to do: carry itself out,
and say which control signals it needs. It does not say *how* — each instruction answers
that for itself.

```cpp
class Instruction {
public:
    virtual ~Instruction() {}
    virtual void           execute(CPU& cpu) = 0;
    virtual ControlSignals signals() const   = 0;
};
```
<sub>src/core/Instruction.h:65</sub>

Fifteen classes derive from it. Here is one, in full:

**What this does.** The whole of the STORE instruction: read the register, put the address
in the MAR and the value in the MDR, then write it to memory. Everything a STORE is,
contained in one place.

```cpp
void StoreInstruction::execute(CPU& cpu) {
    Word v = cpu.registers().readGP(rs_);
    cpu.registers().mar().write(operand_);
    cpu.registers().mdr().write(v);
    cpu.memory().write(operand_.raw(), v);
    cpu.setAluActivity(v, Word(0), v);
}
```
<sub>src/core/Instruction.cpp:126</sub>

And here is the entire execute stage of the processor:

**What this does.** Runs whatever instruction is currently loaded. The CPU does not ask
what kind it is, and contains no instruction-specific code at all.

```cpp
current_->execute(*this);
```
<sub>src/core/CPU.cpp:163</sub>

**Why this is the important line in the project.** Adding a sixteenth instruction means
writing one new class. The CPU is not touched. With a chain of tests, every addition edits
the same growing function — the one place most likely to break, and the one hardest to
test. Polymorphism is not here to demonstrate polymorphism; it is here because the
alternative does not scale.

---

## Decision 3 · Objects that clean up after themselves

Our containers are built from nodes allocated by hand. Something has to free them, and
relying on the programmer to remember is how leaks happen.

**What this does.** The destructor. When a list goes out of scope this runs on its own and
walks the chain, deleting every node.

```cpp
~LinkedList() { clear(); }
```
<sub>src/ds/LinkedList.h:44</sub>

**What this does.** The walk itself. Each node's `next` is saved before the node is
deleted, because reading it afterwards would be reading freed memory.

```cpp
void clear() {
    Node* n = head_;
    while (n != 0) { Node* nx = n->next; delete n; n = nx; }
    head_ = tail_ = 0;
    size_ = 0;
}
```
<sub>src/ds/LinkedList.h:121</sub>

**What this does.** The copy constructor. Copying a list builds new nodes rather than
sharing the originals — otherwise two lists would point at the same memory, and whichever
was destroyed first would leave the other holding freed pointers.

```cpp
LinkedList(const LinkedList& other) : head_(0), tail_(0), size_(0) {
    for (Node* n = other.head_; n != 0; n = n->next) pushBack(n->data);
}
```
<sub>src/ds/LinkedList.h:32</sub>

Constructor, destructor and copy constructor together mean **an object is responsible for
its own memory for its whole life**. No caller has to remember anything.

---

## Decision 4 · Failing without crashing

A simulator runs programs written by a user, and users make mistakes. A typo should
produce a message, not a crash.

**What this does.** One base type for every error the simulator can raise. `kind()` lets
the catcher name the specific problem without needing a separate catch block for each.

```cpp
class SimulatorException : public std::exception {
protected:
    std::string message_;
public:
    explicit SimulatorException(const std::string& msg) : message_(msg) {}
    virtual const char* what() const throw() { return message_.c_str(); }
    virtual std::string kind() const { return "SimulatorException"; }
};
```
<sub>src/core/Exceptions.h:16</sub>

Seven specific errors derive from it — invalid opcode, stack underflow, stack overflow,
index out of range, invalid register, assembly error, divide by zero.

**What this does.** Catches all of them in one place, at the top of the program. Because
they share a base, a single catch block handles every kind while still reporting which one
occurred.

```cpp
catch (const SimulatorException& e) {
    std::cout << "  [" << e.kind() << "] " << e.what() << "\n";
}
```
<sub>src/main.cpp:342</sub>

So a mistyped register produces:

```
[AssemblyErrorException] Assembly error: line 4: 'R9' is not a register
```

The user is told what is wrong and where. The simulator keeps running.

---

## Decision 5 · Making the code read like the thing it describes

**What this does.** Teaches the machine word to behave like a number, so the ALU can be
written as arithmetic rather than as a chain of function calls.

```cpp
Word operator+(const Word& o) const { return Word(static_cast<u16>(value_ + o.value_)); }
Word operator-(const Word& o) const { return Word(static_cast<u16>(value_ - o.value_)); }
Word operator&(const Word& o) const { return Word(static_cast<u16>(value_ & o.value_)); }
```
<sub>src/core/Word.h:44</sub>

`a + b` instead of `add16(a, b)`. A small thing on one line, but the ALU is full of them,
and code that reads like what it describes is code whose mistakes are easier to see.

---

## How to explain CO1 in one minute

1. **The real-world problem:** a processor's workings are invisible, so students memorise
   the cycle without seeing it. Our solution is a simulator you can step through.
2. **It is a big program** — 4,102 lines — and object orientation is what keeps it
   manageable.
3. **Each class owns its own state.** The ALU deliberately owns *nothing*, which is why it
   can be tested alone. Memory is private, so every access passes one function — which is
   how the display knows what to highlight.
4. **Fifteen instructions share one interface.** The processor's entire execute stage is
   `current_->execute(*this)`. Adding an instruction means adding one class and touching
   nothing else.
5. **Objects clean up after themselves** — constructor, destructor and copy constructor,
   so no caller has to remember to free anything.
6. **Errors report instead of crashing.** One exception hierarchy, caught once, naming the
   line that caused it.